In [0]:
# ============================================
# Stage 4.1: Generate DimDate with PySpark
# ============================================

from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# 定义日期范围
start_date = "2011-01-01"
end_date   = "2014-12-31"

# 生成日期序列
date_df = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{start_date}'), 
        to_date('{end_date}'), 
        interval 1 day
    )) AS FullDate
""")

# 添加衍生字段
dim_date = (date_df
    .withColumn("DateKey", F.date_format("FullDate", "yyyyMMdd").cast("int"))
    .withColumn("Year", F.year("FullDate"))
    .withColumn("Quarter", F.quarter("FullDate"))
    .withColumn("Month", F.month("FullDate"))
    .withColumn("MonthName", F.date_format("FullDate", "MMMM"))
    .withColumn("Week", F.weekofyear("FullDate"))
    .withColumn("DayOfMonth", F.dayofmonth("FullDate"))
    .withColumn("DayOfWeek", F.dayofweek("FullDate"))
    .withColumn("DayName", F.date_format("FullDate", "EEEE"))
    .withColumn("IsWeekend", 
        F.when(F.dayofweek("FullDate").isin(1, 7), 1).otherwise(0))
    .select("DateKey", "FullDate", "Year", "Quarter", "Month", "MonthName",
            "Week", "DayOfMonth", "DayOfWeek", "DayName", "IsWeekend")
)

# 预览
display(dim_date.limit(10))

DateKey,FullDate,Year,Quarter,Month,MonthName,Week,DayOfMonth,DayOfWeek,DayName,IsWeekend
20110101,2011-01-01,2011,1,1,January,52,1,7,Saturday,1
20110102,2011-01-02,2011,1,1,January,52,2,1,Sunday,1
20110103,2011-01-03,2011,1,1,January,1,3,2,Monday,0
20110104,2011-01-04,2011,1,1,January,1,4,3,Tuesday,0
20110105,2011-01-05,2011,1,1,January,1,5,4,Wednesday,0
20110106,2011-01-06,2011,1,1,January,1,6,5,Thursday,0
20110107,2011-01-07,2011,1,1,January,1,7,6,Friday,0
20110108,2011-01-08,2011,1,1,January,1,8,7,Saturday,1
20110109,2011-01-09,2011,1,1,January,1,9,1,Sunday,1
20110110,2011-01-10,2011,1,1,January,2,10,2,Monday,0


In [0]:
# 写入 DimDate 到 Curated Zone (Delta Lake 格式)
dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/DimDate/")

print("✅ DimDate written to Curated Zone")

✅ DimDate written to Curated Zone


In [0]:
# 读取验证
df_check = spark.read.format("delta").load(
    "abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/DimDate/"
)

print(f"Total rows: {df_check.count()}")
# 期望：1461

df_check.show(5)

Total rows: 1461
+--------+----------+----+-------+-----+---------+----+----------+---------+---------+---------+
| DateKey|  FullDate|Year|Quarter|Month|MonthName|Week|DayOfMonth|DayOfWeek|  DayName|IsWeekend|
+--------+----------+----+-------+-----+---------+----+----------+---------+---------+---------+
|20110101|2011-01-01|2011|      1|    1|  January|  52|         1|        7| Saturday|        1|
|20110102|2011-01-02|2011|      1|    1|  January|  52|         2|        1|   Sunday|        1|
|20110103|2011-01-03|2011|      1|    1|  January|   1|         3|        2|   Monday|        0|
|20110104|2011-01-04|2011|      1|    1|  January|   1|         4|        3|  Tuesday|        0|
|20110105|2011-01-05|2011|      1|    1|  January|   1|         5|        4|Wednesday|        0|
+--------+----------+----+-------+-----+---------+----+----------+---------+---------+---------+
only showing top 5 rows


In [0]:
# 查看 Delta 表的版本历史
display(spark.sql("DESCRIBE HISTORY delta.`abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/DimDate/`"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-09-27T06:44:54.000Z,140877466452824,yonggang.chen@lincolnuni.ac.nz,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(3935446796285527),fa2a4fdf-6353-4f4a-a72b-69b10deb5e73,0927-054326-5fb7c09z-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1461, numOutputBytes -> 15974)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
